In [6]:
import json
import re


folder = "../results/range/"
input_file = folder + "range_short.txt"
output_file = folder + "range_short.json"


with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()


def parse_float_list(text):
    return [float(x.strip()) for x in text.split(",")]


blocks = re.split(r"=+\s*Epoch\s+(\d+)\s*=+", text)

epochs = []

for i in range(1, len(blocks), 2):
    epoch = int(blocks[i])
    block = blocks[i + 1]

    data = {"epoch": epoch}

    # --------------------------------------------------
    # Main metrics
    # --------------------------------------------------

    match = re.search(r"Train loss:\s*([-\deE+.]+)", block)
    if match:
        data["train_loss"] = float(match.group(1))

    match = re.search(r"NLL on val set:\s*([-\deE+.]+)", block)
    if match:
        data["val_nll"] = float(match.group(1))

    match = re.search(r"NLL on test set:\s*([-\deE+.]+)", block)
    if match:
        data["test_nll"] = float(match.group(1))

    match = re.search(r"Posterior NLL on val:\s*([-\deE+.]+)", block)
    if match:
        data["posterior_val_nll"] = float(match.group(1))

    match = re.search(r"Posterior NLL on train:\s*([-\deE+.]+)", block)
    if match:
        data["posterior_train_nll"] = float(match.group(1))

    match = re.search(r"Prior NLL on train:\s*([-\deE+.]+)", block)
    if match:
        data["prior_train_nll"] = float(match.group(1))

    # --------------------------------------------------
    # Layer statistics
    # --------------------------------------------------

    match = re.search(
        r"Layered KL per layer:\s*\[([^\]]+)\]",
        block,
    )
    if match:
        data["layered_kl"] = parse_float_list(match.group(1))

    match = re.search(
        r"Layer budget per layer:\s*\[([^\]]+)\]",
        block,
    )
    if match:
        data["layer_budget"] = parse_float_list(match.group(1))

    match = re.search(
        r"Layer residual per layer:\s*\[([^\]]+)\]",
        block,
    )
    if match:
        data["layer_residual"] = parse_float_list(match.group(1))

    match = re.search(
        r"lambda_lower per layer:\s*\[([^\]]+)\]",
        block,
    )
    if match:
        data["lambda_lower"] = parse_float_list(match.group(1))

    match = re.search(
        r"lambda_upper per layer:\s*\[([^\]]+)\]",
        block,
    )
    if match:
        data["lambda_upper"] = parse_float_list(match.group(1))

    match = re.search(
        r"lambda_\s*per layer:\s*([-\deE+.]+)",
        block,
    )
    if match:
        data["lambda"] = float(match.group(1))

    # --------------------------------------------------
    # Reconstruction / KL constraint
    # --------------------------------------------------

    match = re.search(
        r"reconstruction=([-\deE+.]+)\s+kl_constraint=([-\deE+.]+)",
        block,
    )
    if match:
        data["reconstruction"] = float(match.group(1))
        data["kl_constraint"] = float(match.group(2))

    # --------------------------------------------------
    # Scrambled NLL
    # -----------------------------------Med fokus på linjär algebra---------------

    match = re.search(
        r"Scrabbled NLL on train:\s*([-\deE+.]+),\s*"
        r"Scrabbled NLL on val:\s*([-\deE+.]+)",
        block,
    )
    if match:
        data["scrabbled_train_nll"] = float(match.group(1))
        data["scrabbled_val_nll"] = float(match.group(2))

    # --------------------------------------------------
    # KL target (appears after previous epoch)
    # --------------------------------------------------

    match = re.search(
        r"KL target set to\s*([-\deE+.]+)",
        block,
    )
    if match:
        data["kl_target"] = float(match.group(1))

    epochs.append(data)




with open(output_file, "w", encoding="utf-8") as f:
    json.dump(epochs, f, indent=4)

print(f"Parsed {len(epochs)} epochs")
print(f"Saved to {output_file}")

Parsed 151 epochs
Saved to ../results/range/range_short.json
